In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats

matplotlib.rcParams["font.family"] = "DejaVu Sans"

def gen_normal(n):
    return np.random.normal(0.0, 1.0, n)

def gen_cauchy(n):
    return stats.cauchy.rvs(loc=0.0, scale=1.0, size=n)

def gen_laplace(n):
    return np.random.laplace(0.0, 1.0 / np.sqrt(2), n)

def gen_poisson(n):
    return np.random.poisson(10, n)

def gen_uniform(n):
    return np.random.uniform(-np.sqrt(3), np.sqrt(3), n)

DISTRIBUTIONS = {
    "normal":  ("Нормальное N(0,1)",      gen_normal),
    "cauchy":  ("Коши C(0,1)",            gen_cauchy),
    "laplace": ("Лапласа L(0,1/√2)",      gen_laplace),
    "poisson": ("Пуассона P(10)",         gen_poisson),
    "uniform": ("Равномерное U(-√3,√3)",  gen_uniform),
}

def count_outliers(sample: np.ndarray) -> int:
    q1 = np.quantile(sample, 0.25)
    q3 = np.quantile(sample, 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(np.sum((sample < lower) | (sample > upper)))


def outlier_fraction(sample: np.ndarray) -> float:
    return count_outliers(sample) / len(sample)


def mean_outlier_fraction(generator, n: int, m: int = 1000) -> float:
    fractions = [outlier_fraction(generator(n)) for _ in range(m)]
    return float(np.mean(fractions))

def plot_boxplots(dist_key: str, dist_name: str, generator, ns: list):
    samples = [generator(n) for n in ns]

    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(
        samples,
        labels=[f"n = {n}" for n in ns],
        vert=True,
        patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.6),
        medianprops=dict(color="red", linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        flierprops=dict(marker=".", markersize=4, color="gray", alpha=0.6,
                        label="Выброс"),
    )
    ax.set_title(f"Боксплот Тьюки\nРаспределение: {dist_name}", fontsize=12, weight="bold")
    ax.set_ylabel("Значение", fontsize=10)
    ax.set_xlabel("Объём выборки", fontsize=10)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    fname = f"lab3_boxplot_{dist_key}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Сохранён: {fname}")

def plot_outlier_fractions(all_fractions: dict, ns: list):
    dist_keys = list(all_fractions.keys())
    dist_labels = [DISTRIBUTIONS[k][0] for k in dist_keys]
    x = np.arange(len(dist_keys))
    width = 0.35
    colors = ["#4C72B0", "#C44E52"]

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, (n, col) in enumerate(zip(ns, colors)):
        vals = [all_fractions[k][n] for k in dist_keys]
        bars = ax.bar(x + (i - 0.5) * width, vals, width=width,
                      label=f"n = {n}", color=col, alpha=0.8,
                      edgecolor="black", linewidth=0.5)
        # Подписи значений над столбцами
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.002,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(dist_labels, fontsize=9, rotation=12)
    ax.set_ylabel("Средняя доля выбросов", fontsize=10)
    ax.set_title("Средняя доля выбросов по правилу Тьюки (1000 выборок)", fontsize=12, weight="bold")
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.set_ylim(0, max(
        all_fractions[k][n] for k in dist_keys for n in ns
    ) * 1.2 + 0.01)

    plt.tight_layout()
    fname = "lab3_outlier_fractions.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Сохранён: {fname}")

def main():
    ns_box  = [20, 100]
    m = 1000 

    all_fractions = {}

    for dist_key, (dist_name, generator) in DISTRIBUTIONS.items():
        print(f"\n{dist_name}")

        plot_boxplots(dist_key, dist_name, generator, ns_box)

        fractions = {}
        for n in ns_box:
            frac = mean_outlier_fraction(generator, n, m)
            fractions[n] = frac
            print(f"  n = {n:>4}: средняя доля выбросов = {frac:.4f} ({frac*100:.2f}%)")

        all_fractions[dist_key] = fractions
        
    plot_outlier_fractions(all_fractions, ns_box)



if __name__ == "__main__":
    main()